# BIRD Benchmark — VKG Query-Agent Eval (LLM judges + codeBased exec metrics)

Runs the **BIRD** benchmark's natural-language questions through the deployed
**ontology_query** agent (VKG path, `QueryRuntimeArn`) and scores each answer
**two ways, entirely server-side** via AgentCore Batch Evaluations:

1. **LLM-as-Judge (SESSION)** — `GoalSuccess`, `FinalAnswerFaithfulness`,
   `SqlGrounded` from `agents/shared/eval_judges.py` (VKG family), identical to
   `5_ontology_queryagent_ac_eval.ipynb`.
2. **codeBased exec metrics (TRACE)** — `ExecutionAccuracy` (EX), `RewardVES`
   (R-VES), `SoftF1`, computed by the **BenchmarkEvalHandler** Lambda
   (`BenchmarkEvalHandlerArn`). Each is registered as a `codeBased` evaluator
   pointing at the SAME Lambda; the Lambda branches on the evaluator name. It
   extracts the agent's predicted SQL from the trace spans, reads the **gold
   SQL** from the reference input, executes both on Athena, and returns EX /
   R-VES / Soft-F1.

### Prerequisites
- **`11_bird_semantic_layer.ipynb` built the per-DB VKG layers** (ids
  `vkg-bird_<db>`). This notebook filters BIRD rows to DBs that have a completed
  layer and invokes the agent with `ontologyId = f"vkg-{row.db_id}"`.
- **`BenchmarkEvalHandler` Lambda deployed** (the AgentCore stack's Part-A
  codeBased evaluator; CFN output `BenchmarkEvalHandlerArn`).
- The BIRD DBs are ingested (manifest under `data/eval/benchmarks/bird/`).

### Gold-packing (READ THIS — the dual-scoring contract)
The codeBased Lambda reads gold from `expectedResponse.text` as JSON
`{"gold_sql", "db_id", "catalog"}`. The nb5 LLM judges, by contrast, read the
expected **NL answer** from `{assertions}` (threaded via
`FINAL_ANSWER_ASSERTION_PREFIX`). These want DIFFERENT things in the same field.

**Resolution:** we pack the gold JSON into each scenario's `expected_response`
(via `parse_multiturn_row` → `build_scenarios`, which puts it on the final
`Turn.expected_response` → the batch API's TRACE `expectedResponse.text`). That
is exactly what the codeBased Lambda reads, so **EX / R-VES / Soft-F1 are
correct**. BIRD rows carry gold SQL, **not** a prose answer, so the same packed
JSON also becomes the LLM judge's `FINAL_ANSWER_ASSERTION_PREFIX` reference —
which makes `FinalAnswerFaithfulness` **weak/uninformative for BIRD** (it is
comparing the agent's NL answer to a JSON blob). That is an accepted trade-off:
**the exec metrics are the primary BIRD signal**; `GoalSuccess` and `SqlGrounded`
(which don't depend on a gold NL answer) remain meaningful.


In [1]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [2]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


✓ AWS Credentials Verified:
  Profile: default
  Account: ...1429
  User/Role: srbhsrkr-Isengard
  Region: us-east-1


In [3]:
import re

def _redact_account_ids(obj):
    """Recursively mask AWS account IDs in ARN strings to their last 4 digits (********NNNN)."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(\d{8})(\d{4})', r'********\2', obj)
    return obj


def _mask_acct(text):
    """Mask AWS account IDs (12-digit runs) to their last 4 digits — e.g. ********2087 —
    in any string (ARNs, bucket names, plain ids). Never emits the full account number."""
    import re as __re
    if not isinstance(text, str):
        return text
    return __re.sub(r'(\d{8})(\d{4})', r'********\2', text)


In [4]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# Invoke the JWT-inbound runtime with a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    # The deployed function carries the stage infix (e.g. semantic-layer-dev-mcp-tools);
    # try both the bare and the '-dev-' variant, mirroring the auth-stack lookup above.
    cfg = None
    for _fn_name in (f'{PROJECT_NAME}-mcp-tools', f'{PROJECT_NAME}-dev-mcp-tools'):
        try:
            cfg = lam.get_function_configuration(FunctionName=_fn_name)
            break
        except lam.exceptions.ResourceNotFoundException:
            continue
    if cfg is None:
        raise RuntimeError(f'mcp-tools Lambda not found for project {PROJECT_NAME}')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

✓ OAuth M2M runtime invoker ready


## 2. Load BIRD rows + resolve the built VKG layers

In [5]:
# ── Load BIRD rows and keep only those with a built VKG layer + gold SQL ─────
import sys
sys.path.insert(0, '..')
from pathlib import Path
from agents.shared.benchmark_adapter import load_rows
from scripts.benchmarks.ingest_common import Manifest

BENCHMARK = 'bird'
BENCH_DIR = Path(f'../data/eval/benchmarks/{BENCHMARK}')
JSONL = BENCH_DIR / f'{BENCHMARK}.jsonl'

# Manifest maps each db_id (= namespace "bird_<db>") -> catalog / database /
# tables. We use each row's manifest entry to build the codeBased gold payload
# (the Lambda needs the federated catalog + Athena database to execute gold SQL).
_manifest_path = None
for _cand in (BENCH_DIR / 'manifest.json', BENCH_DIR / 'deploy_manifest.json'):
    if _cand.exists():
        _manifest_path = _cand
        break
if _manifest_path is None:
    raise FileNotFoundError(
        f"No BIRD manifest under {BENCH_DIR}. Run scripts/benchmarks/"
        f"deploy_datasets.py --benchmark bird first."
    )
manifest = Manifest.load(path=_manifest_path)
_ingested_ids = {e.db_id for e in manifest.entries}

all_rows = load_rows(path=JSONL)

# ── Which BIRD DBs have a COMPLETED VKG layer (built by notebook 11)? ─────────
# nb11 seeds/builds layers with the deterministic id `vkg-<namespace>` where
# namespace == the BIRD db_id (e.g. "bird_superhero"). We scan DynamoDB for
# completed vkg-bird_* layers so this notebook is self-sufficient (it does NOT
# require nb11 to have %store'd anything in this kernel).
METADATA_TABLE = os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
_meta_table = session.resource('dynamodb', region_name=region).Table(METADATA_TABLE)


def _completed_bird_vkg_layers() -> set:
    """Return the set of db_ids (namespaces) that have a completed vkg-bird_* layer."""
    done, scan_kw = set(), {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'vkg-bird_', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        for it in page.get('Items', []):
            lid = it['id']                       # vkg-bird_<db>
            if lid.startswith('vkg-'):
                done.add(lid[len('vkg-'):])      # -> bird_<db> (== row.db_id)
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    return done


_built_db_ids = _completed_bird_vkg_layers()
print(f"✓ Completed VKG layers for {len(_built_db_ids)} BIRD db(s): "
      f"{sorted(_built_db_ids) or '(none)'}")
if not _built_db_ids:
    raise RuntimeError(
        "No completed vkg-bird_* layers found. Run 11_bird_semantic_layer.ipynb "
        "first (it builds one VKG layer per BIRD db)."
    )

# Keep rows that are (a) ingested, (b) have a built VKG layer, and (c) carry gold
# SQL (execution-scoreable). Advisory/meta rows with no gold SQL cannot be scored.
rows = [r for r in all_rows
        if r.db_id in _ingested_ids and r.db_id in _built_db_ids and r.gold_sql]

# ── PER-DB SCOPING (BIRD_DB) — why this notebook is run once per database ─────
# All 500 rows in ONE batch does not fit: 123 of them are non-convergent rows that
# now stop at the Tier 3 worker budget (~795s each) instead of the client timeout,
# so even at raised concurrency the single cell approaches the runner's 14h
# CELL_TIMEOUT — which is how the 2026-08-07 run ended with 500 rows invoked and
# ZERO scores written (the cell died before the persist step).
#
# Setting BIRD_DB=<db_id> (or a comma-separated list) scopes this run to one
# database. The largest DB is 66 rows, so each run finishes in well under an hour
# and PERSISTS ITS OWN RESULTS FILE. A crash then costs one database, not all 11.
# Unset/empty BIRD_DB keeps the original all-DBs behaviour.
BIRD_DB = os.environ.get('BIRD_DB', '').strip()
if BIRD_DB:
    _wanted = {d.strip() for d in BIRD_DB.split(',') if d.strip()}
    _unknown = _wanted - _built_db_ids
    if _unknown:
        # Fail loudly: a typo'd db id would otherwise silently score 0 rows.
        raise ValueError(
            f"BIRD_DB names db(s) with no completed VKG layer: {sorted(_unknown)}. "
            f"Available: {sorted(_built_db_ids)}"
        )
    rows = [r for r in rows if r.db_id in _wanted]
    print(f"  ▸ BIRD_DB={BIRD_DB} — scoped to {len(rows)} row(s) in {len(_wanted)} db(s)")

MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    rows = rows[:MAX_ROWS]
    print(f"  ⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(rows)}-row slice (0 = all)")

if not rows:
    raise RuntimeError(
        "No scoreable BIRD rows (need an ingested + VKG-built db AND gold SQL)."
    )
print(f"✓ {len(rows)} scoreable BIRD row(s) across "
      f"{len({r.db_id for r in rows})} db(s)")
pd.set_option('display.max_colwidth', None)
display(pd.DataFrame([{'db_id': r.db_id, 'difficulty': r.difficulty,
                       'question': r.question} for r in rows]).head(20))


✓ Completed VKG layers for 11 BIRD db(s): ['bird_california_schools', 'bird_card_games', 'bird_codebase_community', 'bird_debit_card_specializing', 'bird_european_football_2', 'bird_financial', 'bird_formula_1', 'bird_student_club', 'bird_superhero', 'bird_thrombosis_prediction', 'bird_toxicology']
  ▸ BIRD_DB=bird_california_schools — scoped to 30 row(s) in 1 db(s)
✓ 30 scoreable BIRD row(s) across 1 db(s)


,db_id,difficulty,question
0,bird_california_schools,simple,How many schools with an average score in Math greater than 400 in the SAT test are exclusively virtual?
1,bird_california_schools,simple,Please list the codes of the schools with a total enrollment of over 500.
2,bird_california_schools,moderate,"Among the schools with an SAT excellence rate of over 0.3, what is the highest eligible free rate for students aged 5-17?"
3,bird_california_schools,simple,"Rank schools by their average score in Writing where the score is greater than 499, showing their charter numbers."
4,bird_california_schools,moderate,List the names of schools with more than 30 difference in enrollements between K-12 and ages 5-17? Please also give the full street adress of the schools.
5,bird_california_schools,moderate,Give the names of the schools with the percent eligible for free meals in K-12 is more than 0.1 and test takers whose test score is greater than or equal to 1500?
6,bird_california_schools,moderate,"Name schools in Riverside which the average of average math score for SAT is grater than 400, what is the funding type of these schools?"
7,bird_california_schools,moderate,State the names and full communication address of high schools in Monterey which has more than 800 free or reduced price meals for ages 15-17?
8,bird_california_schools,moderate,"What is the average score in writing for the schools that were opened after 1991 or closed before 2000? List the school names along with the score. Also, list the communication number of the schools if there is any."
9,bird_california_schools,challenging,"Consider the average difference between K-12 enrollment and 15-17 enrollment of schools that are locally funded, list the names and DOC type of schools which has a difference above this average."


## 3. Resolve the ontology_query runtime + the codeBased evaluator Lambda

In [6]:
# ── Look up the ontology_query runtime ARN + the BenchmarkEvalHandler Lambda ──
# BOTH come from the {PROJECT_NAME}-agentcore stack outputs (try the -dev suffix
# too). The VKG query agent's output key is `QueryRuntimeArn` (NOT
# `OntologyQueryRuntimeArn`); the codeBased exec-metrics Lambda is
# `BenchmarkEvalHandlerArn` (export `<project>-benchmark-eval-handler-arn`).
cfn = session.client('cloudformation', region_name=region)
outputs = None
for _stack in (f'{PROJECT_NAME}-agentcore', f'{PROJECT_NAME}-dev-agentcore'):
    try:
        outputs = cfn.describe_stacks(StackName=_stack)['Stacks'][0]['Outputs']
        stack_name = _stack
        break
    except Exception:  # noqa: BLE001 — try the -dev fallback stack name
        continue
if outputs is None:
    raise RuntimeError(f"AgentCore stack not found for project {PROJECT_NAME}")
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

ONTOLOGY_QUERY_RUNTIME_ARN = output_map['QueryRuntimeArn']
# AgentCore's batch path does NOT pass evaluatorName to the codeBased Lambda,
# so each metric is a SEPARATE function with EVAL_METRIC baked into its env.
# Resolve one ARN per metric (output keys BenchmarkEvalHandler<Metric>Arn).
BENCHMARK_EVAL_LAMBDA_ARNS = {
    'ExecutionAccuracy': output_map['BenchmarkEvalHandlerExecutionAccuracyArn'],
    'RewardVES': output_map['BenchmarkEvalHandlerRewardVESArn'],
    'SoftF1': output_map['BenchmarkEvalHandlerSoftF1Arn'],
}
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = ONTOLOGY_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"✓ From stack '{stack_name}':")
print(f"  Query Runtime ARN     : {_mask_acct(ONTOLOGY_QUERY_RUNTIME_ARN)}")
print(f"  Agent ID              : {agent_id}")
for _m, _arn in BENCHMARK_EVAL_LAMBDA_ARNS.items():
    print(f"  BenchmarkEval {_m:<17}: {_mask_acct(_arn)}")


✓ From stack 'semantic-layer-dev-agentcore':
  Query Runtime ARN     : arn:aws:bedrock-agentcore:us-east-1:********1429:runtime/semantic_layer_dev_ontology_query-OIEQK6HvjQ
  Agent ID              : semantic_layer_dev_ontology_query-OIEQK6HvjQ
  BenchmarkEval ExecutionAccuracy: arn:aws:lambda:us-east-1:********1429:function:semantic-layer-dev-agentc-BenchmarkEvalHandlerExec-3b2Sp63CsEUO
  BenchmarkEval RewardVES        : arn:aws:lambda:us-east-1:********1429:function:semantic-layer-dev-agentc-BenchmarkEvalHandlerRewa-hGTKlWXKo1rY
  BenchmarkEval SoftF1           : arn:aws:lambda:us-east-1:********1429:function:semantic-layer-dev-agentc-BenchmarkEvalHandlerSoft-zBMV4OhMt14z


## 4. Create the evaluators — LLM judges (SESSION) + codeBased exec metrics (TRACE)

Two evaluator families, both scored server-side:

| Evaluator | Kind | Level | Source |
|:--|:--|:--|:--|
| `GoalSuccess` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `FinalAnswerFaithfulness` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `SqlGrounded` | LLM-as-Judge | SESSION | `eval_judges` (VKG) |
| `bird_ExecutionAccuracy` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |
| `bird_RewardVES` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |
| `bird_SoftF1` | codeBased | TRACE | `BenchmarkEvalHandler` Lambda |

> The codeBased evaluators branch on the evaluator NAME (the Lambda matches the
> canonical substrings `ExecutionAccuracy` / `RewardVES` / `SoftF1`), so a
> `bird_` prefix is fine. All three point at the SAME Lambda ARN.

> **`Builtin.GoalSuccessRate` is intentionally omitted** — nb5 documents that the
> AWS-managed builtin mis-scores this deterministic-graph agent (it mistakes an
> intermediate intent/SPARQL span for the answer). The custom `GoalSuccess`
> SESSION judge replaces it. (If you want the builtin as a cross-check, append
> `'Builtin.GoalSuccessRate'` to `ALL_EVALUATOR_IDS` in the next cell — read it
> skeptically.)


In [7]:
# ── LLM-as-Judge (SESSION) evaluators — VKG family, from eval_judges ─────────
try:
    from agents.shared.eval_judges import (
        get_or_create_custom_judges, find_evaluator_id_by_name, JUDGE_MODEL_ID,
    )
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_judges import (
        get_or_create_custom_judges, find_evaluator_id_by_name, JUDGE_MODEL_ID,
    )

_cp = session.client('bedrock-agentcore-control', region_name=region)

# ── EVALUATOR REUSE (BIRD_EVAL_SUFFIX) ───────────────────────────────────────
# This notebook is run ONCE PER DATABASE (see the BIRD_DB cell above), so a fresh
# random suffix would mint 6 brand-new evaluators every run — 66 across the 11
# BIRD databases, on top of the ~270 this account already carries (AgentCore never
# garbage-collects evaluators). Worse, per-DB scores would then come from
# *different* evaluator objects and would not be strictly poolable.
#
# A STABLE suffix makes evaluator resolution idempotent: the first run creates the
# six, every later run looks them up by name. Bump BIRD_EVAL_SUFFIX whenever a
# judge prompt or a Lambda ARN changes, so a NEW generation is minted rather than
# silently reusing evaluators built on the old rubric.
BIRD_EVAL_SUFFIX = os.environ.get('BIRD_EVAL_SUFFIX', 'bird_vkg_v1').strip()
if not BIRD_EVAL_SUFFIX:
    raise ValueError("BIRD_EVAL_SUFFIX must be non-empty — it is the reuse key.")
print(f"Evaluator generation suffix: {BIRD_EVAL_SUFFIX}")

# family='vkg' → [GoalSuccess, FinalAnswerFaithfulness, SqlGrounded] (byte-identical
# to nb5's judges). GoalSuccess is the custom replacement for the un-editable
# Builtin.GoalSuccessRate on this deterministic-graph agent.
GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID = get_or_create_custom_judges(
    control_client=_cp, family='vkg', name_suffix=BIRD_EVAL_SUFFIX,
)
CUSTOM_JUDGE_IDS = [GOAL_SUCCESS_ID, ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID]
print("✓ Custom SESSION LLM-as-Judge evaluators resolved:")
print(f"  GoalSuccess             : {GOAL_SUCCESS_ID}")
print(f"  FinalAnswerFaithfulness : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             : {SQL_GROUNDED_ID}")


# ── codeBased exec-metric (TRACE) evaluators — one per metric, same Lambda ───
# The AgentCore CreateEvaluator codeBased config is
#   {'codeBased': {'lambdaConfig': {'lambdaArn': <arn>, 'lambdaTimeoutInSeconds': N}}}
# level=TRACE (one score per trace). We register three, whose names carry the
# canonical metric substrings the Lambda branches on (ExecutionAccuracy / RewardVES
# / SoftF1). Resolved by name under BIRD_EVAL_SUFFIX so per-DB runs reuse them.
#
# Lambda run_sql polls up to 120s PER query and runs gold THEN predicted, so give
# the codeBased invocation headroom (max allowed is 300s).
_LAMBDA_TIMEOUT_S = 280


def _get_or_create_codebased_evaluator(*, metric_name: str) -> str:
    """Resolve one codeBased exec-metric evaluator (TRACE) by name, creating if absent.

    Parameters:
        metric_name: The metric substring the Lambda branches on — one of
            'ExecutionAccuracy', 'RewardVES', 'SoftF1'.
    Returns:
        The evaluator's evaluatorId (the existing one if a same-named evaluator
        is already registered, else a freshly created one).
    """
    evaluator_name = f'bird_{metric_name}_{BIRD_EVAL_SUFFIX}'
    existing = find_evaluator_id_by_name(
        control_client=_cp, evaluator_name=evaluator_name,
    )
    if existing is not None:
        print(f"  ↩ reusing {evaluator_name}")
        return existing
    resp = _cp.create_evaluator(
        evaluatorName=evaluator_name,
        level='TRACE',
        evaluatorConfig={
            'codeBased': {
                'lambdaConfig': {
                    # Per-metric function ARN — EVAL_METRIC in its env
                    # tells the shared handler which metric to score.
                    'lambdaArn': BENCHMARK_EVAL_LAMBDA_ARNS[metric_name],
                    'lambdaTimeoutInSeconds': _LAMBDA_TIMEOUT_S,
                }
            }
        },
    )
    return resp['evaluatorId']


EX_ID = _get_or_create_codebased_evaluator(metric_name='ExecutionAccuracy')
RVES_ID = _get_or_create_codebased_evaluator(metric_name='RewardVES')
SOFTF1_ID = _get_or_create_codebased_evaluator(metric_name='SoftF1')
CODEBASED_IDS = [EX_ID, RVES_ID, SOFTF1_ID]
print("\n✓ codeBased exec-metric evaluators resolved (TRACE, one Lambda per metric):")
print(f"  bird_ExecutionAccuracy : {EX_ID}")
print(f"  bird_RewardVES         : {RVES_ID}")
print(f"  bird_SoftF1            : {SOFTF1_ID}")

# All evaluator ids the batch job scores. LLM judges (SESSION) + exec metrics
# (TRACE) — the batch API applies each at its own level from one flat list.
ALL_EVALUATOR_IDS = [*CUSTOM_JUDGE_IDS, *CODEBASED_IDS]

# Display-name map for the results tables (event names drop the trailing -<hash>).
_EVAL_NAME = {
    GOAL_SUCCESS_ID: 'GoalSuccess', ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
    SQL_GROUNDED_ID: 'SqlGrounded', EX_ID: 'ExecutionAccuracy',
    RVES_ID: 'RewardVES', SOFTF1_ID: 'SoftF1',
}
_EVAL_NAME.update({_eid.rsplit('-', 1)[0]: _disp for _eid, _disp in list(_EVAL_NAME.items())})


Evaluator generation suffix: bird_vkg_v1


✓ Custom SESSION LLM-as-Judge evaluators resolved:
  GoalSuccess             : GoalSuccess_bird_vkg_v1-WAq9qL7bTU
  FinalAnswerFaithfulness : FinalAnswerFaithfulness_bird_vkg_v1-lybtFK4iWP
  SqlGrounded             : SqlGrounded_bird_vkg_v1-6UGFZiEJf6
  ↩ reusing bird_ExecutionAccuracy_bird_vkg_v1


  ↩ reusing bird_RewardVES_bird_vkg_v1
  ↩ reusing bird_SoftF1_bird_vkg_v1

✓ codeBased exec-metric evaluators resolved (TRACE, one Lambda per metric):
  bird_ExecutionAccuracy : bird_ExecutionAccuracy_bird_vkg_v1-nGDcPw79pB
  bird_RewardVES         : bird_RewardVES_bird_vkg_v1-5PvVgqAp5Q
  bird_SoftF1            : bird_SoftF1_bird_vkg_v1-sp6BmXC7kP


## 5. Build the Dataset (one scenario per BIRD row) + agent_invoker

Each BIRD row is one `PredefinedScenario` (its own session), bound to its DB's
VKG layer via `ontologyId = f"vkg-{row.db_id}"`.

**Gold-packing:** each row's `expected_response` is set to
`json.dumps({"gold_sql", "db_id", "catalog"})` (via `parse_multiturn_row` +
`build_scenarios`). The batch API delivers that as the TRACE
`expectedResponse.text` — which the codeBased Lambda reads for EX / R-VES /
Soft-F1. (It also becomes the LLM judge's `FINAL_ANSWER_ASSERTION_PREFIX`
reference; see the intro for why `FinalAnswerFaithfulness` is weak on BIRD.)


In [8]:
# Per-row chat-stream invoker + dataset builder (mirrors nb5, but binds ontologyId
# PER ROW and packs the codeBased gold into expected_response).
import time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# ── Pack the codeBased gold into each row's Expected_Answer ──────────────────
# The codeBased Lambda reads expectedResponse.text = json.dumps({gold_sql, db_id,
# catalog}); build_scenarios threads a row's Expected_Answer onto the final
# Turn.expected_response (→ the TRACE expectedResponse.text). So we set
# Expected_Answer to the packed gold. db_id (= namespace bird_<db>) is the Athena
# database the Lambda executes gold SQL in; catalog is the federated S3 Tables
# catalog (per-row from the manifest entry). One scenario per row.
_specs = []
SCENARIO_LAYER = {}        # scenario_id -> ontologyId (vkg-<db_id>)
SCENARIO_ROW = {}          # scenario_id -> the BenchmarkRow (for the results file)
for i, row in enumerate(rows):
    entry = manifest.entry(db_id=row.db_id)
    gold_payload = json.dumps({
        'gold_sql': row.gold_sql,
        'db_id': row.db_id,          # == namespace bird_<db> == Athena database
        'catalog': entry.catalog,    # federated S3 Tables catalog id
    })
    spec = parse_multiturn_row(
        {'Natural_Language_Question': row.question, 'Expected_Answer': gold_payload},
        index=i,
    )
    layer_id = f'vkg-{row.db_id}'    # SAME id nb11 built
    SCENARIO_LAYER[spec['scenario_id']] = layer_id
    SCENARIO_ROW[spec['scenario_id']] = row
    _specs.append(spec)

# build_scenarios takes ONE ontology_id (stamped into scenario metadata), but BIRD
# rows span DBs, so call it PER spec with that row's layer id and concatenate. The
# real ontologyId binding happens in agent_invoker (below) via build_chat_payload.
scenarios = []
for spec in _specs:
    scenarios.extend(build_scenarios(
        [spec], ontology_id=SCENARIO_LAYER[spec['scenario_id']],
        simulated_enabled=False))
dataset = Dataset(scenarios=scenarios)
print(f"✓ Dataset: {len(dataset.scenarios)} BIRD scenario(s) (one per row)")


AGENT_RUNS = {}            # keyed by run_key(session_id, turn_idx)


def _scenario_id_for_session(sess: str) -> str:
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Invoke the VKG query agent for one BIRD row via the chat-stream path.

    Binds ontologyId to the row's per-DB VKG layer (vkg-<db_id>), parses the SSE
    body, and records the predicted SQL + cost/latency into AGENT_RUNS.
    """
    sid = invoker_input.session_id
    scenario_id = _scenario_id_for_session(sid)
    ontology_id = SCENARIO_LAYER.get(scenario_id, '')
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=ontology_id, turn_idx=turn_idx,
                                 user_id="bird-eval", source="eval")
    payload["mode"] = "vkg"                    # VKG path (default is semantic-rag)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(ONTOLOGY_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001 — one row failure must not abort the batch
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "scenario_id": scenario_id, "turn_idx": turn_idx,
        "ontology_id": ontology_id, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        "executed_sql": parsed.get("executed_sql", ""), "degraded": parsed.get("degraded"),
        "rows": parsed.get("rows", []) or [], "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "answer" if parsed.get("sql") else ("clarify" if parsed.get("clarification") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{scenario_id} {ontology_id}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))


✓ Dataset: 30 BIRD scenario(s) (one per row)


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent once per scenario (via `agent_invoker`),
waits `ingestion_delay_seconds` for CloudWatch span ingestion, submits ONE
`StartBatchEvaluation` over all six evaluators (SESSION judges + TRACE exec
metrics), and polls to completion. All scoring is in-service.


In [9]:
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")
print(f"Evaluators   : {ALL_EVALUATOR_IDS}")
print(f"Scenarios    : {len(dataset.scenarios)}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    ingestion_delay_seconds=420,   # give spans (incl. the SQL-exec span) time to land
)

batch_runner = BatchEvaluationRunner(region=region)

# ── Concurrency (BIRD_CONCURRENCY) ───────────────────────────────────────────
# Raised 3 → 8. The agent is synchronous and a worst-case row now runs to the
# Tier 3 worker budget (3 rounds x 240s + ~75s ≈ 795s, just inside the invoker's
# 900s urlopen timeout), so serialized wall-clock is what killed the 500-row
# single-batch run. Concurrency is bounded by the agent runtime's own scaling and
# by Bedrock model TPM, not by this notebook — 8 keeps the largest per-DB run (66
# rows) to roughly 66/8 x 795s ≈ 1.8h worst case, comfortably inside the runner's
# CELL_TIMEOUT with the batch-polling stage still ahead of it.
BIRD_CONCURRENCY = int(os.environ.get('BIRD_CONCURRENCY', '8'))
# batchEvaluationName must match [a-zA-Z][a-zA-Z0-9_]{0,47} — at most 48 chars and
# ONLY letters/digits/underscore (no hyphens). The previous f-string broke both
# rules: it truncated the scope to 40 chars (so 'bird_debit_card_specializing'
# produced a 52-char name) and mapped ',' to '-'. StartBatchEvaluation then failed
# with ValidationException AFTER every row had already been invoked, discarding a
# whole database's agent work. Budget the scope by what remains after the fixed
# prefix and the 9-char uuid suffix, and sanitize to the allowed charset.
_BATCH_NAME_MAX = 48
_batch_prefix = 'bird_vkg_bench_'
_batch_suffix = f'_{uuid.uuid4().hex[:8]}'
_batch_scope = ''.join(ch if ch.isalnum() else '_' for ch in (BIRD_DB or 'all'))
_batch_budget = _BATCH_NAME_MAX - len(_batch_prefix) - len(_batch_suffix)
_batch_name = _batch_prefix + _batch_scope[:_batch_budget] + _batch_suffix
# Fail loudly rather than letting the service reject it after the invocation spend.
if len(_batch_name) > _BATCH_NAME_MAX or not _batch_name[0].isalpha():
    raise ValueError(f'invalid batchEvaluationName: {_batch_name!r}')

cfg = BatchEvaluationRunConfig(
    batch_evaluation_name=_batch_name,
    description="BIRD VKG query-agent eval: LLM judges (SESSION) + codeBased "
                "exec metrics EX/R-VES/Soft-F1 (TRACE).",
    evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
    data_source=batch_data_source,
    max_concurrent_scenarios=BIRD_CONCURRENCY,  # synchronous agent, up to 900s/row
    polling_timeout_seconds=5400,  # VKG rows are multi-phase + the codeBased Lambda runs 2 Athena queries/row
    polling_interval_seconds=30,
)
print(f"Concurrency  : {BIRD_CONCURRENCY}")

print(f"\n=== {cfg.batch_evaluation_name} ===")
print("Invoking the agent per row, then evaluating server-side...")
batch_result = batch_runner.run_dataset_evaluation(
    config=cfg, dataset=dataset, agent_invoker=agent_invoker)
print(f"\n✓ Batch status: {batch_result.status}")
if getattr(batch_result, 'agent_invocation_failures', None):
    print(f"⚠ Agent invocation failures: {len(batch_result.agent_invocation_failures)}")
if batch_result.error_details:
    print(f"⚠ Error details: {batch_result.error_details}")


SERVICE_NAME : semantic_layer_dev_ontology_query-OIEQK6HvjQ.DEFAULT
LOG GROUPS   : aws/spans, /aws/bedrock-agentcore/runtimes/semantic_layer_dev_ontology_query-OIEQK6HvjQ-DEFAULT
Evaluators   : ['GoalSuccess_bird_vkg_v1-WAq9qL7bTU', 'FinalAnswerFaithfulness_bird_vkg_v1-lybtFK4iWP', 'SqlGrounded_bird_vkg_v1-6UGFZiEJf6', 'bird_ExecutionAccuracy_bird_vkg_v1-nGDcPw79pB', 'bird_RewardVES_bird_vkg_v1-5PvVgqAp5Q', 'bird_SoftF1_bird_vkg_v1-sp6BmXC7kP']
Scenarios    : 30


Concurrency  : 8

=== bird_vkg_bench_bird_california_schools_d492f20b ===
Invoking the agent per row, then evaluating server-side...


  OK [gt-row-00 vkg-bird_california_schools] 6.07s . none


  OK [gt-row-01 vkg-bird_california_schools] 51.83s . answer


  OK [gt-row-06 vkg-bird_california_schools] 53.11s . answer


  OK [gt-row-08 vkg-bird_california_schools] 69.38s . answer


... (23 outputs left out)

  OK [gt-row-28 vkg-bird_california_schools] 201.71s . answer


  OK [gt-row-22 vkg-bird_california_schools] 286.05s . answer


  OK [gt-row-23 vkg-bird_california_schools] 302.88s . answer



✓ Batch status: COMPLETED_WITH_ERRORS
⚠ Error details: ['18 of 30 sessions failed during batch evaluation.']


## 7. Results — LLM-judge aggregates + exec-metric (EX / R-VES / Soft-F1) aggregates

1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results`.
   The three codeBased evaluators' `average_score` **are** the mean EX / R-VES /
   Soft-F1 over the scored rows; the three judges' are the mean pass rates.
2. **Per-row evaluator scores** — `fetch_evaluation_events()`, joined by session id.
3. **Per-row agent detail** — predicted SQL / cost / latency from `AGENT_RUNS`.


In [10]:
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f"Batch ID : {batch_result.batch_evaluation_id}")
print(f"Status   : {batch_result.status}")

# ── 1. Aggregate per-evaluator scores (LLM judges + exec metrics) ────────────
agg_rows = []
_agg_by_name = {}
ev = batch_result.evaluation_results
if ev is not None:
    print(f"\nSessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (stats.average_score if stats and stats.average_score is not None else None)
        name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
        _agg_by_name[name] = round(avg, 4) if avg is not None else None
        agg_rows.append({
            'Evaluator': name,
            'AvgScore': f"{avg:.3f}" if avg is not None else None,
            'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0,
        })
else:
    print("\n⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

# ── 2. Per-row evaluator events ──────────────────────────────────────────────
def _event_field(e: dict, key: str):
    """Read an event field from the top level or the nested 'attributes' map."""
    if key in e:
        return e[key]
    return (e.get('attributes', {}) or {}).get(key)

events = []
for _attempt in range(6):
    try:
        events = batch_runner.fetch_evaluation_events(batch_result)
        if events:
            break
    except (LookupError, ValueError) as exc:
        print(f"  per-row events not ready yet ({exc}); retrying in 20s...")
    time.sleep(20)

event_rows = []
for e in events:
    sess = _event_field(e, 'session.id') or _event_field(e, 'gen_ai.session.id')
    name = _event_field(e, 'gen_ai.evaluation.name')
    sid = _scenario_id_for_session(sess)
    _row = SCENARIO_ROW.get(sid)
    event_rows.append({
        'scenario_id': sid,
        'db_id': getattr(_row, 'db_id', ''),
        'difficulty': getattr(_row, 'difficulty', '') or '',
        'question': getattr(_row, 'question', ''),
        'evaluator': _EVAL_NAME.get(name, name),
        'score': _event_field(e, 'gen_ai.evaluation.score.value'),
        'label': _event_field(e, 'gen_ai.evaluation.score.label'),
        'explanation': str(_event_field(e, 'gen_ai.evaluation.explanation') or ''),
    })
df_events = pd.DataFrame(event_rows)
print(f"\nPer-row evaluator events: {len(df_events)}")

# ── Per-difficulty + per-DB EX/R-VES/Soft-F1 from the exec-metric events ─────
# Both breakdowns carry a COUNT alongside the mean. Per-DB runs are pooled after
# the fact into one BIRD-wide EX, and pooling means requires the per-slice n —
# averaging 11 per-DB averages would weight a 30-row DB the same as a 66-row one.
_EXEC_NAMES = {'ExecutionAccuracy', 'RewardVES', 'SoftF1'}
by_diff, by_db = {}, {}
for r in event_rows:
    if r['evaluator'] not in _EXEC_NAMES or not isinstance(r['score'], (int, float)):
        continue
    by_diff.setdefault(r['difficulty'] or 'unknown', {}).setdefault(
        r['evaluator'], []).append(r['score'])
    by_db.setdefault(r['db_id'] or 'unknown', {}).setdefault(
        r['evaluator'], []).append(r['score'])


def _mean_with_n(scores_by_metric: dict) -> dict:
    """Reduce {metric: [scores]} to {metric: mean, metric_n: count} for pooling.

    Parameters:
        scores_by_metric: Mapping of metric name -> list of numeric scores.
    Returns:
        Dict with a rounded mean per metric plus a '<metric>_n' count per metric.
    """
    out = {}
    for metric, values in scores_by_metric.items():
        out[metric] = round(sum(values) / len(values), 4)
        out[f'{metric}_n'] = len(values)
    return out


exec_by_difficulty = {band: _mean_with_n(m) for band, m in by_diff.items()}
exec_by_db = {db: _mean_with_n(m) for db, m in by_db.items()}

# ── 2b. Telemetry validity gate — CHECK THIS BEFORE READING ANY MEAN ─────────
# EX / R-VES / Soft-F1 are SPAN-DERIVED: the evaluator Lambda reads the agent's SQL
# out of the session spans. A row whose trace has not landed in CloudWatch by the
# time the evaluator runs is UNMEASURABLE, and the Lambda returns
# errorCode=TelemetryUnavailable for it instead of scoring 0.0. Such rows therefore
# drop OUT of each metric's n rather than silently deflating its mean — which makes
# COVERAGE the first number to read: a mean computed over 60% of the rows says
# nothing about the benchmark. On the 2026-08-09 11-DB run, 142/500 rows were in
# this state and pooled EX read 0.0448 where the intact subset read 0.0625.
#
# BUT an unscored row has TWO possible causes and AgentCore reports both as
# score=null, because it collapses every evaluator errorCode to a null score:
#
#   TelemetryUnavailable — the trace had not landed. A REAL measurement gap.
#   GoldExecutionFailed  — the benchmark's OWN gold SQL does not run on Athena.
#                          A PERMANENT dataset defect; the row is unanswerable in
#                          principle and belongs in NO denominator.
#
# Netting the second class out is what makes the floor meaningful. Measured on the
# 2026-08-10 run: 53 rows unscored → reported 89.4% coverage, but probing all 53
# gold queries showed 51 were invalid SQL (26 TYPE_MISMATCH from BIRD's all-varchar
# columns, 11 FUNCTION_NOT_FOUND, 10 EXPRESSION_NOT_AGGREGATE, 4 other) and only 2
# were true telemetry loss. Real coverage was 99.55%, and all eight databases
# "below the floor" were an artifact of the dataset. Judging the pipeline on 89.4%
# would have sent us optimising ingestion delay for a problem that did not exist.
#
# scripts/benchmarks/validate_gold_sql.py writes the sidecar this reads. When it is
# absent we fall back to the raw denominator and SAY SO — never silently.
_gold_validity_path = f'../data/eval/benchmarks/bird/gold_validity.json'
_unanswerable_ids = set()
_gold_validity_note = ''
if os.path.exists(_gold_validity_path):
    with open(_gold_validity_path) as _fh:
        _gv = json.load(_fh)
    for _db, _bad in (_gv.get('invalid') or {}).items():
        for _sid in _bad:
            _unanswerable_ids.add((_db, _sid))
    _gold_validity_note = (
        f"gold_validity.json probed {_gv.get('probed_at', '?')}: "
        f"{_gv.get('n_invalid', 0)}/{_gv.get('n_rows', 0)} rows unanswerable"
    )
else:
    _gold_validity_note = (
        'gold_validity.json ABSENT — coverage is computed against ALL rows, so '
        'rows whose gold SQL cannot run on Athena are counted as telemetry loss. '
        'Run scripts/benchmarks/validate_gold_sql.py to separate the two.'
    )

# Rows in THIS run that are unanswerable because their gold SQL is invalid.
_run_unanswerable = {
    (r.db_id, sid) for sid, r in SCENARIO_ROW.items()
    if (r.db_id, sid) in _unanswerable_ids
}
_n_rows = len(rows)
_n_answerable = _n_rows - len(_run_unanswerable)

_scored_per_metric = {
    m: len([r for r in event_rows
            if r['evaluator'] == m and isinstance(r['score'], (int, float))])
    for m in sorted(_EXEC_NAMES)
}
# Coverage against ANSWERABLE rows — the honest denominator. Also keep the raw
# figure so a reader can see what the dataset cost us.
_coverage = {m: (round(n / _n_answerable, 4) if _n_answerable else None)
             for m, n in _scored_per_metric.items()}
_coverage_raw = {m: (round(n / _n_rows, 4) if _n_rows else None)
                 for m, n in _scored_per_metric.items()}
_min_coverage = min([c for c in _coverage.values() if c is not None], default=0.0)
# Configurable so a deliberately degraded run can still be recorded; the DEFAULT
# is strict because an unnoticed 28% hole is exactly how the 0.0448 figure got
# quoted as a real accuracy for two days.
_COVERAGE_FLOOR = float(os.environ.get('BIRD_MIN_TELEMETRY_COVERAGE', '0.95'))
telemetry_coverage = {
    'n_rows': _n_rows,
    'n_answerable': _n_answerable,
    'n_unanswerable_gold': len(_run_unanswerable),
    'unanswerable_scenario_ids': sorted(sid for _db, sid in _run_unanswerable),
    'gold_validity_note': _gold_validity_note,
    'scored_rows_per_metric': _scored_per_metric,
    'coverage_per_metric': _coverage,
    'coverage_per_metric_raw': _coverage_raw,
    'min_coverage': _min_coverage,
    'coverage_floor': _COVERAGE_FLOOR,
    'coverage_ok': _min_coverage >= _COVERAGE_FLOOR,
}
print(f"\n{_gold_validity_note}")
if _run_unanswerable:
    print(f"Excluding {len(_run_unanswerable)} unanswerable row(s) from the "
          f"denominator: {_n_rows} invoked → {_n_answerable} answerable")
print(f"Telemetry coverage (floor={_COVERAGE_FLOOR:.2f}): "
      f"min={_min_coverage:.4f} " + "  ".join(
          f"{m}={_coverage[m]}({_scored_per_metric[m]}/{_n_answerable})"
          for m in sorted(_scored_per_metric)))
if not telemetry_coverage['coverage_ok']:
    print(f"\n⚠⚠ TELEMETRY LOSS: only {_min_coverage:.1%} of ANSWERABLE rows were "
          f"measurable (floor {_COVERAGE_FLOOR:.0%}). The exec-metric means below "
          f"describe ONLY the measured subset and must NOT be quoted as a benchmark "
          f"score. Raise ingestion_delay_seconds (applied AFTER the last invocation) "
          f"or cut per-row latency, then re-run.")

# ── 3. Per-row agent detail (predicted SQL / cost / latency) ─────────────────
agent_detail = []
for rec in AGENT_RUNS.values():
    agent_detail.append({
        'scenario_id': rec['scenario_id'], 'ontology_id': rec['ontology_id'],
        'question': rec['message'], 'agent_sql': rec['agent_sql'],
        'executed_sql': rec['executed_sql'], 'degraded': rec['degraded'],
        'wall_clock_s': rec['wall_clock_s'], 'error': rec['invoke_error'],
    })
df_agent = pd.DataFrame(agent_detail)

# ── Persist ──────────────────────────────────────────────────────────────────
# The filename carries the DB scope so the 11 per-DB runs land as 11 clearly
# identifiable files (a bare timestamp made them indistinguishable at a glance).
_scope_tag = (BIRD_DB or 'all').replace(',', '-')
combined_file = (f"../data/eval/results/"
                 f"bird_vkg_benchmark_eval_{_scope_tag}_{timestamp}.json")
_db_ids = sorted({r.db_id for r in rows})
combined = {
    'benchmark': 'bird',
    'agent': 'ontology_query_agent',
    'agent_id': agent_id,
    'runtime_arn': ONTOLOGY_QUERY_RUNTIME_ARN,
    'benchmark_eval_lambda_arns': BENCHMARK_EVAL_LAMBDA_ARNS,
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status),
    # Run scope — what this file covers, so per-DB runs are poolable without
    # re-deriving anything from the filename.
    'bird_db_scope': BIRD_DB or 'all',
    'db_ids': _db_ids,
    'max_concurrent_scenarios': BIRD_CONCURRENCY,
    'evaluator_suffix': BIRD_EVAL_SUFFIX,
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'llm_judges': {'GoalSuccess': GOAL_SUCCESS_ID,
                   'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                   'SqlGrounded': SQL_GROUNDED_ID},
    'exec_metric_evaluators': {'ExecutionAccuracy': EX_ID, 'RewardVES': RVES_ID,
                               'SoftF1': SOFTF1_ID},
    # Headline: the LLM-judge aggregates AND the exec-metric (EX/R-VES/Soft-F1) aggregates.
    'llm_judge_aggregates': {k: _agg_by_name.get(k) for k in
                             ('GoalSuccess', 'FinalAnswerFaithfulness', 'SqlGrounded')},
    'exec_metric_aggregates': {k: _agg_by_name.get(k) for k in
                               ('ExecutionAccuracy', 'RewardVES', 'SoftF1')},
    'exec_metrics_by_difficulty': exec_by_difficulty,
    'exec_metrics_by_db': exec_by_db,
    'telemetry_coverage': telemetry_coverage,
    'aggregate_summaries': agg_rows,
    'per_row_events': event_rows,
    'agent_detail': agent_detail,
    'n_rows': len(rows),
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"\n✓ Wrote {combined_file}")

print("\n=== Aggregate per-evaluator scores ===")
display(df_agg)
_ex = combined['exec_metric_aggregates']
print(f"\nHeadline exec metrics (scope={BIRD_DB or 'all'}, n={len(rows)}): "
      f"EX={_ex.get('ExecutionAccuracy')}  R-VES={_ex.get('RewardVES')}  "
      f"Soft-F1={_ex.get('SoftF1')}")
if exec_by_difficulty:
    print("\nExec metrics by difficulty:")
    for band in sorted(exec_by_difficulty):
        m = exec_by_difficulty[band]
        print(f"  {band:>12}: EX={m.get('ExecutionAccuracy')}  "
              f"R-VES={m.get('RewardVES')}  Soft-F1={m.get('SoftF1')}  "
              f"(n={m.get('ExecutionAccuracy_n')})")
if exec_by_db:
    print("\nExec metrics by db:")
    for db in sorted(exec_by_db):
        m = exec_by_db[db]
        print(f"  {db:>32}: EX={m.get('ExecutionAccuracy')}  "
              f"R-VES={m.get('RewardVES')}  Soft-F1={m.get('SoftF1')}  "
              f"(n={m.get('ExecutionAccuracy_n')})")
print("\n=== Per-row evaluator scores ===")
display(df_events)
print("=== Per-row agent detail ===")
display(df_agent)

# Raise LAST: the results file is already on disk, so a coverage failure is loud
# (non-zero notebook status → recorded in the driver's failed_runs) without
# destroying the run's evidence.
if not telemetry_coverage['coverage_ok']:
    raise RuntimeError(
        f"Telemetry coverage {_min_coverage:.4f} is below the floor "
        f"{_COVERAGE_FLOOR:.2f}: {_scored_per_metric} of {_n_rows} rows measurable. "
        f"Exec metrics for this run are not a valid benchmark score. "
        f"Results were still written to {combined_file}."
    )


Batch ID : bird_vkg_bench_bird_california_schools_d492f20b-1b70b1c52d
Status   : COMPLETED_WITH_ERRORS

Sessions: completed=12 failed=18 total=30



Per-row evaluator events: 175

gold_validity.json probed 2026-08-10T04:27:34.397495+00:00: 51/500 rows unanswerable
Excluding 6 unanswerable row(s) from the denominator: 30 invoked → 24 answerable
Telemetry coverage (floor=0.95): min=0.9583 ExecutionAccuracy=0.9583(23/24)  RewardVES=0.9583(23/24)  SoftF1=0.9583(23/24)

✓ Wrote ../data/eval/results/bird_vkg_benchmark_eval_bird_california_schools_20261105_115319.json

=== Aggregate per-evaluator scores ===


,Evaluator,AvgScore,Evaluated,Failed
0,SoftF1,NaN,23,6
1,SqlGrounded,1.000,16,13
2,ExecutionAccuracy,NaN,23,6
3,RewardVES,NaN,23,6
4,GoalSuccess,0.730,15,14
5,FinalAnswerFaithfulness,0.750,16,13



Headline exec metrics (scope=bird_california_schools, n=30): EX=None  R-VES=None  Soft-F1=None

Exec metrics by difficulty:
   challenging: EX=0.0  R-VES=0.0  Soft-F1=0.0  (n=4)
      moderate: EX=0.1538  R-VES=0.1154  Soft-F1=0.1538  (n=13)
        simple: EX=0.1667  R-VES=0.125  Soft-F1=0.1667  (n=6)

Exec metrics by db:
           bird_california_schools: EX=0.1304  R-VES=0.0978  Soft-F1=0.1304  (n=23)

=== Per-row evaluator scores ===


,scenario_id,db_id,difficulty,question,evaluator,score,label,explanation
0,gt-row-00,bird_california_schools,simple,How many schools with an average score in Math greater than 400 in the SAT test are exclusively virtual?,NaN,NaN,NaN,
1,gt-row-01,bird_california_schools,simple,Please list the codes of the schools with a total enrollment of over 500.,GoalSuccess,0.0,fail,"The gold SQL defines ""total enrollment"" as the sum of 'Enrollment (K-12)' and 'Enrollment (Ages 5-17)', filtered where that sum exceeds 500, and returns CDSCode. The agent's executed query only filtered on 'Enrollment (K-12)' > 500, ignoring the Ages 5-17 component entirely. This produces a different set of qualifying schools and a different count (the gold definition would include schools where K-12 alone is under 500 but the sum with Ages 5-17 exceeds 500, which the agent's query misses, and could exclude some the agent includes under different circumstances). The final-answer record (the assistant's natural language reply stating '5,362 schools with a total K-12 enrollment over 500...') reflects this incorrect single-column enrollment definition rather than the gold's summed definition. Since the underlying computation and resulting codes/count do not match the gold SQL's intended definition of 'total enrollment', the final answer is factually inconsistent with the expected result, even though the correct columns (CDSCode) were returned. Therefore the assertion that the final answer matches the gold SQL is violated."
2,gt-row-01,bird_california_schools,simple,Please list the codes of the schools with a total enrollment of over 500.,FinalAnswerFaithfulness,0.0,fail,"The gold SQL defines 'total enrollment' as the sum of Enrollment (K-12) and Enrollment (Ages 5-17) exceeding 500, joined via schools.CDSCode. The agent's query only filtered on Enrollment (K-12) > 500, ignoring the Ages 5-17 enrollment component entirely. This is a materially different condition that would yield a different set of schools and likely a different count than the gold query intends. The final answer reports 5,362 schools with examples based on K-12 enrollment alone, which does not correctly reflect the 'total enrollment' definition in the expected gold query. Thus the final answer is inconsistent with the expected result."
3,gt-row-01,bird_california_schools,simple,Please list the codes of the schools with a total enrollment of over 500.,SqlGrounded,1.0,pass,"The result columns 'CDSCODE1m1' and 'ENROLLMENT (K-12)1m15' correspond to frpm.cdscode and frpm.""enrollment (k-12)"", both of which are defined in the retrieved ontology context (Frpm/cdscode and Frpm/enrollmentK12, mapsToColumn 'frpm.cdscode' and 'frpm.enrollment (k-12)'). No executed SQL text is shown explicitly, but the result schema aligns directly with the ontology-mapped columns, and no hallucinated table/column is evident. The filter 'enrollment > 500' uses a column present in the ontology. All referenced fields trace back to the Frpm class mappings, satisfying the grounding invariant."
4,gt-row-01,bird_california_schools,simple,Please list the codes of the schools with a total enrollment of over 500.,ExecutionAccuracy,0.0,FAIL,"metric=ex value=0.0000 (pred_rows=5362 in 1477ms, gold_rows=7806 in 2785ms) | pred_row0=['01100170109835', '1087.0'] gold_row0=['01100170109835']"
...,...,...,...,...,...,...,...,...
170,gt-row-12,bird_california_schools,moderate,"What is the complete address of the school with the lowest excellence rate? Indicate the Street, City, Zip and State.",FinalAnswerFaithfulness,1.0,pass,"The agent's final answer gives Street: 2607 Myrtle Street, City: Oakland, Zip: 94607-3415, State: CA for the school with the lowest excellence rate (NumGE1500/NumTstTakr). This corresponds to the known BIRD benchmark answer for this exact question, which matches the semantics of the gold SQL (ordering by SAT excellence rate ascending with NumTstTakr>0 handling for valid rates). The retrieved row's identifiers, city, st

=== Per-row agent detail ===


,scenario_id,ontology_id,question,agent_sql,executed_sql,degraded,wall_clock_s,error
0,gt-row-00,vkg-bird_california_schools,How many schools with an average score in Math greater than 400 in the SAT test are exclusively virtual?,,,None,6.07,None
1,gt-row-01,vkg-bird_california_schools,Please list the codes of the schools with a total enrollment of over 500.,SELECT ?cdscode WHERE {\n ?f a <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/Frpm> ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/Frpm/cdscode> ?cdscode ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/Frpm/enrollmentK12> ?enroll .\n FILTER(xsd:decimal(?enroll) > 500)\n},"SELECT R1.CDSCODE AS ""CDSCODE1m1"", R1.""ENROLLMENT (K-12)"" AS ""ENROLLMENT (K-12)1m15""\nFROM ""bird_california_schools"".""frpm"" R1\nWHERE ((CAST(CASE WHEN REGEXP_LIKE(CAST(R1.""ENROLLMENT (K-12)"" AS VARCHAR), '^[-+]?[0-9]*\.?[0-9]*$') THEN CAST(CAST(R1.""ENROLLMENT (K-12)"" AS VARCHAR) AS DECIMAL(38, 18)) ELSE NULL END AS DECIMAL(38, 18)) > CAST(500 AS DECIMAL(38, 18))) AND R1.""ENROLLMENT (K-12)"" IS NOT NULL AND R1.CDSCODE IS NOT NULL)",None,51.83,None
2,gt-row-06,vkg-bird_california_schools,"Name schools in Riverside which the average of average math score for SAT is grater than 400, what is the funding type of these schools?","SELECT ?schoolName ?fundingtype WHERE {\n ?s a <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School> ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/county> ""Riverside"" ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/school> ?schoolName ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/fundingtype> ?fundingtype ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/cdscode> ?cds .\n ?sat a <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/SatScore> ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/SatScore/cds> ?cds ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/SatScore/avgscrmath> ?math .\n}\nGROUP BY ?schoolName ?fundingtype\nHAVING (AVG(?math) > 400)","SELECT R4.""FUNDINGTYPE1m69"" AS ""FUNDINGTYPE1m69"", R4.""SCHOOL1m85"" AS ""SCHOOL1m85"", R4.""avg1"" AS ""avg1""\nFROM (SELECT R1.FUNDINGTYPE AS ""FUNDINGTYPE1m69"", R1.SCHOOL AS ""SCHOOL1m85"", AVG(R2.AVGSCRMATH) AS ""avg1""\nFROM ""bird_california_schools"".""schools"" R1, ""bird_california_schools"".""satscores"" R2\nWHERE (R1.SCHOOL IS NOT NULL AND R1.FUNDINGTYPE IS NOT NULL AND R2.AVGSCRMATH IS NOT NULL AND R1.CDSCODE = R2.CDS AND 'Riverside' = R1.COUNTY)\nGROUP BY R1.FUNDINGTYPE, R1.SCHOOL\n) R4\nWHERE (R4.""avg1"" > CAST(400 AS DECIMAL(38, 18)))\n",None,53.11,None
3,gt-row-08,vkg-bird_california_schools,"What is the average score in writing for the schools that were opened after 1991 or closed before 2000? List the school names along with the score. Also, list the communication number of the schools if there is any.","SELECT ?schoolName ?avgscrwrite ?phone WHERE {\n ?school a <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School> ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/school> ?schoolName ;\n <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/cdscode> ?cds .\n OPTIONAL { ?school <http://bird-bird_california_schools-ontology-20261002-161105/ontology/vkg-bird_california_schools/School/phone> ?phone }\n OPTIONAL { ?school <http://bird-bi

## Summary

This notebook runs the **BIRD** benchmark through the deployed **ontology_query**
(VKG) agent and scores each answer **server-side, two ways**:

1. **LLM judges (SESSION)** — `GoalSuccess`, `FinalAnswerFaithfulness`,
   `SqlGrounded` from `eval_judges` (VKG family), identical to nb5.
2. **codeBased exec metrics (TRACE)** — `bird_ExecutionAccuracy` (EX),
   `bird_RewardVES` (R-VES), `bird_SoftF1`, three `codeBased` evaluators pointing
   at the `BenchmarkEvalHandler` Lambda (`BenchmarkEvalHandlerArn`), which extracts
   the predicted SQL from the trace, reads the packed gold from
   `expectedResponse.text`, and executes both on Athena.

### Gold-packing / dual-scoring trade-off
The gold JSON `{gold_sql, db_id, catalog}` is packed into each scenario's
`expected_response` (→ TRACE `expectedResponse.text`), which is what the codeBased
Lambda reads — so **EX / R-VES / Soft-F1 are the correct, primary BIRD signal**.
BIRD carries no prose answer, so that same packed JSON is what the LLM
`FinalAnswerFaithfulness` judge sees as its expected answer → **weak for BIRD**
(accepted). `GoalSuccess` and `SqlGrounded` don't depend on a gold NL answer and
remain meaningful.

### Layer binding
`ontologyId = f"vkg-{row.db_id}"` — the exact ids `11_bird_semantic_layer.ipynb`
built. Rows are filtered to DBs with a completed `vkg-bird_*` layer.

### Notes
- **R-VES caveat** (as in the client-side benchmark notebooks): Athena engine time
  is not comparable to BIRD's SQLite reference timing, so read R-VES as a
  within-arm relative signal. **EX** and **Soft-F1** are the portable metrics.
- Results → `../data/eval/results/bird_vkg_benchmark_eval_*.json` (LLM-judge
  aggregates + exec-metric aggregates + per-difficulty EX/R-VES/Soft-F1 +
  per-row detail).
